# E18B — 단계별 실행 노트북

질문 위치 다층 K와 L13 단일층 K를 비교하고 C 확률은 고정합니다.
기존 E18B와 같은 분할·모델·PCA·판독기·평가 기준입니다. 코드 포장만 바꿨으며 새 결과 폴더에 저장합니다.

**실행 순서:** 설치 → Drive → 기존 특징 파일 확인 → 함수 정의 → 데이터 준비 → 모델 인증 → 특징 추출 → 학습 및 평가 → 결과 확인.

코드는 모두 일반 Python 셀입니다. 각 함수 셀은 정의만 하며, 실제 실행은 단계 끝의 실행 셀에서 수행합니다.
필요한 기존 특징·라벨·기준 결과는 원본 파일 ID로 자동 다운로드하며 해시를 검증합니다. 파일 추가 업로드는 없습니다. Drive와 Hugging Face 인증은 필요할 수 있습니다.


## 1. 환경 설치 — A100 런타임 선택 후 실행

In [ ]:
%pip -q install scikit-learn==1.6.1 transformers==4.57.1 threadpoolctl==3.7.0 gdown joblib


## 2. Google Drive 연결 및 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > A100 GPU'
assert 'A100' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
print('GPU:', torch.cuda.get_device_name(0))


## 3. 기존 특징 자동 준비
원본 E18A 특징을 자동으로 가져옵니다. 비공개 파일은 Google 계정 인증 후 다운로드합니다. 추가 업로드나 경로 설정은 필요하지 않습니다.

In [ ]:
from pathlib import Path
import hashlib
import shutil

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fh:
        for block in iter(lambda: fh.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

_drive_reader = None

def ensure_input(file_id, target, expected_sha, local_candidates=()):
    """정확한 원본만 사용: 로컬 캐시 → 다운로드 → 인증된 Drive 읽기."""
    global _drive_reader
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    for candidate in [target, *map(Path, local_candidates)]:
        if candidate.is_file() and file_sha256(candidate) == expected_sha:
            if candidate.resolve() != target.resolve():
                shutil.copyfile(candidate, target)
            print('입력 확인:', target.name)
            return target
    tmp = target.with_suffix(target.suffix + '.download')
    import gdown
    try:
        gdown.download(id=file_id, output=str(tmp), quiet=False)
    except Exception as exc:
        print('공개 다운로드 불가; 계정 인증으로 가져옵니다:', type(exc).__name__)
    if not tmp.is_file() or file_sha256(tmp) != expected_sha:
        if _drive_reader is None:
            from google.colab import auth
            import google.auth
            from googleapiclient.discovery import build
            auth.authenticate_user()
            credentials, _ = google.auth.default()
            _drive_reader = build('drive', 'v3', credentials=credentials, cache_discovery=False)
        from googleapiclient.http import MediaIoBaseDownload
        try:
            with tmp.open('wb') as fh:
                download = MediaIoBaseDownload(fh, _drive_reader.files().get_media(fileId=file_id), chunksize=8*1024*1024)
                done = False
                while not done:
                    status, done = download.next_chunk()
        except Exception as exc:
            raise RuntimeError('원본 Drive 파일을 읽지 못했습니다. 기존 E18A 파일에 접근 가능한 Google 계정으로 인증한 뒤 이 셀을 재실행하세요. 추가 파일 업로드는 필요 없습니다.') from exc
    if file_sha256(tmp) != expected_sha:
        raise ValueError(f'원본 해시 불일치: {target.name}. 다른 파일을 대신 사용하지 않습니다.')
    tmp.replace(target)
    print('자동 다운로드 및 원본 검증 완료:', target.name)
    return target

INPUT_CACHE = Path('/content/drive/MyDrive/p8s/E18B_input_cache')
OLD_BANK = ensure_input(
    '1rM0vcm-IV8qiYJA2CNmHPf76NsgzvYaO',
    INPUT_CACHE / 'neutral_features_all.npz',
    '9ead3c0c2671bacf727253f95b27021e744ba30d1f2f29b893750b933fae652c',
    local_candidates=['/content/drive/MyDrive/p8s/E18A_earlyK_lateC_v1/neutral_capture/neutral_features_all.npz'],
)
_input_setup_cell = get_ipython().execution_count


## 4. 공통 함수 정의
라벨·질문 단위 분할·프롬프트·평가 함수입니다. 아래 셀들을 순서대로 실행하세요.

In [ ]:
_definition_start = get_ipython().execution_count + 1


In [ ]:
"""E18A: fixed-design, question-grouped K/C four-state diagnostic evaluation.

This module does not modify LLM weights, perform QA generation, or implement steering.
All learned readers are supervised diagnostic probes, not label-free routers.
"""
from __future__ import annotations
import hashlib
import importlib.metadata
import json
import platform
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Mapping

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, log_loss, precision_recall_fscore_support, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

STATE_NAMES = ['K+C+', 'K+C-', 'K-C-', 'K-C+']
MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
MODEL_REVISION = '0e9e39f249a16976918f6564b8830bc894c89659'
SYSTEM_PROMPT = (
    'You are a question answering system. Answer the question based on the given context. '
    'Output the answer string itself and nothing else: no sentence, no explanation, '
    'no punctuation, no quotes. Keep it as short as possible (a name or a term).'
)
SOURCE_FILES = {
    'behavior': dict(id='1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr', name='p8s_behavior_v2.csv',
        sha256='fd37b0afd04972e3834ffa5f0af7f362affcfac8e20325117d37d8de15447c1c'),
    'bank13': dict(id='1GK0xi217u57Wz_kn8ycg5eSlPgjsaFNy', name='L13_neutral_head_bank_all_rows.npz',
        sha256='4ad4a040388d1e45995be7aadc7b91f6dfd1f4091b4fd452da8b4a9036b83802'),
    'oof13': dict(id='1djoU4xPIAnCQjBLrb4a3YYIqStWLM6En', name='OOF_L13_LAST_HIDDEN_PREDICTIONS.csv',
        sha256='2f8f214aa44fe1a6e020edbb1bf3456f9ed30aa147d9bce09b30bfa72150677d'),
}



### Spec

In [ ]:
@dataclass(frozen=True)
class Spec:
    name: str
    mode: str
    keys: tuple[str, ...]
    purpose: str


In [ ]:

SPECS = (
    Spec('M0_L13_joint4', 'joint4', ('L13',), 'E14 L13-last joint four-class reference'),
    Spec('M1_L13_factor', 'factor', ('L13', 'L13'), 'Same-layer K/C factorization control'),
    Spec('M2_L7lastK_L13lastC', 'factor', ('L7', 'L13'), 'PRIMARY: change K reading layer only'),
    Spec('M3_L13qK_L13lastC', 'factor', ('Q13', 'L13'), 'Change K token location only'),
    Spec('M4_L7qK_L13lastC', 'factor', ('Q7', 'L13'), 'Early K plus question-position reading'),
    Spec('M5_L7q_L13last_joint4', 'joint4', ('Q7', 'L13'), 'Two-vector joint decoder'),
    Spec('M6_L13q_L13last_joint4', 'joint4', ('Q13', 'L13'), 'Dimension-matched control for M5'),
)
CONTRASTS = (
    ('M2_L7lastK_L13lastC', 'M1_L13_factor', 'PRIMARY_layer_only'),
    ('M2_L7lastK_L13lastC', 'M0_L13_joint4', 'secondary_vs_original'),
    ('M1_L13_factor', 'M0_L13_joint4', 'factorization_only'),
    ('M3_L13qK_L13lastC', 'M1_L13_factor', 'position_only'),
    ('M4_L7qK_L13lastC', 'M3_L13qK_L13lastC', 'early_layer_at_question'),
    ('M5_L7q_L13last_joint4', 'M6_L13q_L13last_joint4', 'joint_dimension_matched_layer'),
    ('M5_L7q_L13last_joint4', 'M4_L7qK_L13lastC', 'joint_vs_factor_not_parameter_matched'),
)



### sha256_file

In [ ]:
def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


### write_json

In [ ]:
def write_json(path: str | Path, obj) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)


### environment

In [ ]:
def environment() -> dict:
    out = {'python': platform.python_version(), 'platform': platform.platform()}
    for name in ['numpy', 'pandas', 'scikit-learn', 'torch', 'transformers', 'threadpoolctl']:
        try: out[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: out[name] = None
    return out


### freeze_plan

In [ ]:
def freeze_plan(out_dir: str | Path, *, specs=SPECS, bootstrap=2000, seed=42) -> dict:
    """A timestamped local analysis lock, NOT independent/external preregistration."""
    p = Path(out_dir) / 'analysis_plan.json'
    plan = {
        'experiment': 'E18A_EarlyK_LateC_4State_v1',
        'status': 'development_followup_not_independent_confirmation',
        'split': 'E14 fixed question-grouped 5 outer folds, 60% fit/10% UF/10% PC/20% test',
        'state_order': STATE_NAMES, 'K_definition': 'E14 kside_corr; DO NOT use E17 K_strict',
        'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
        'prompt': 'original E11/E14 e11-qfirst, one question/context prompt per row',
        'layer_indexing': 'zero-based decoder block output: layers[7], layers[13]',
        'C_logreg': 0.1, 'solver': 'lbfgs', 'max_iter': 2000, 'seed': seed,
        'standardization': 'fit partition only; no test normalization',
        'binary_threshold': 0.5, 'class_weight': None,
        'methods': [asdict(x) for x in specs], 'contrasts': list(CONTRASTS),
        'bootstrap_replicates': bootstrap,
        'bootstrap_unit': 'question; both C conditions together; conditional on fitted OOF predictions',
        'primary_outcome': 'M2-M1 pooled OOF 4-state macro-F1; paired question CI',
        'no_claims': ['no QA EM improvement measured', 'no causal intervention tested',
                      'not runtime probe-free', 'no guaranteed improvement',
                      'no independent held-out confirmation'],
    }
    canonical = json.loads(json.dumps(plan))
    if p.exists():
        old = json.loads(p.read_text(encoding='utf-8'))
        old.pop('created_utc', None)
        if old != canonical:
            raise ValueError('Analysis plan differs. Use a NEW run directory; do not overwrite results.')
    else:
        write_json(p, {**canonical, 'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())})
    (p.parent / 'analysis_plan.sha256').write_text(sha256_file(p) + '\n')
    return canonical


### cell_ids

In [ ]:
def cell_ids(k, c) -> np.ndarray:
    k, c = np.asarray(k, int), np.asarray(c, int)
    if not np.isin(k, [0, 1]).all() or not np.isin(c, [0, 1]).all():
        raise ValueError('K/C must be binary.')
    return np.where(k == 1, np.where(c == 1, 0, 1), np.where(c == 1, 3, 2)).astype(int)


### load_labels

In [ ]:
def load_labels(behavior: str | Path, oof_reference: str | Path | None = None) -> pd.DataFrame:
    f = pd.read_csv(behavior).reset_index(drop=True)
    for name in ['question', 'ctx_text', 'ctx_label', 'kside_corr']:
        if name not in f or f[name].isna().any(): raise ValueError(f'Missing/empty column: {name}')
    f['k'] = f.kside_corr.map({'K+': 1, 'K-': 0, 'K−': 0})
    f['c'] = f.ctx_label.map({'C+': 1, 'C-': 0, 'C−': 0})
    if f[['k', 'c']].isna().any().any(): raise ValueError('Unknown K/C encoding; no silent relabeling.')
    f[['k', 'c']] = f[['k', 'c']].astype(int)
    f['state'] = cell_ids(f.k, f.c); f['row'] = np.arange(len(f))
    if len(f) != 1224 or f.question.nunique() != 612: raise ValueError('Expected original E14 612Q / 1224 rows.')
    if not f.groupby('question').size().eq(2).all(): raise ValueError('Question pair size must be 2.')
    if not f.groupby('question').k.nunique().eq(1).all(): raise ValueError('K inconsistent across a pair.')
    if not f.groupby('question').c.nunique().eq(2).all(): raise ValueError('Need exactly C+ and C- per question.')
    if oof_reference is not None:
        r = pd.read_csv(oof_reference).sort_values('row')
        if not np.array_equal(r.row, f.row) or not np.array_equal(r.true_state, f.state):
            raise ValueError('E14 stored labels disagree with kside_corr. Stop and reconcile; never replace labels.')
        fold = f.question.map(build_outer_map(f)).to_numpy()
        if not np.array_equal(r.fold, fold): raise ValueError('E14 outer fold mismatch.')
    return f


### build_outer_map

In [ ]:
def build_outer_map(f: pd.DataFrame, seed=42) -> dict:
    q = f[['question', 'k']].drop_duplicates().sort_values('question').reset_index(drop=True)
    rng = np.random.default_rng(seed); mapping = {}
    for cls in (0, 1):
        ids = q.loc[q.k.eq(cls), 'question'].to_numpy().copy(); rng.shuffle(ids)
        mapping.update({s: i % 5 for i, s in enumerate(ids)})
    return mapping


### make_split

In [ ]:
def make_split(f: pd.DataFrame, test_fold: int, seed=42) -> pd.DataFrame:
    fold = f.question.map(build_outer_map(f, seed)).to_numpy()
    role = np.full(len(f), 'fit', object)
    role[fold == test_fold] = 'test'; role[fold == (test_fold + 1) % 5] = 'cal'
    rng = np.random.default_rng(seed + 1000 * test_fold)
    for cls in (0, 1):
        qids = np.array(sorted(set(f.loc[(role == 'cal') & f.k.eq(cls), 'question'])))
        rng.shuffle(qids); half = len(qids) // 2
        role[(role == 'cal') & f.question.isin(qids[:half])] = 'utility_fit'
        role[(role == 'cal') & f.question.isin(qids[half:])] = 'policy_cal'
    sp = f[['row', 'question', 'k', 'c', 'state']].copy()
    sp['outer_fold'] = fold; sp['role'] = role
    if not sp.groupby('question').role.nunique().eq(1).all(): raise ValueError('Group leakage.')
    for r in ['fit', 'utility_fit', 'policy_cal', 'test']:
        if set(sp.loc[sp.role.eq(r), 'state']) != {0, 1, 2, 3}:
            raise ValueError(f'Missing state in fold {test_fold}, role {r}.')
    return sp


### factor_probs

In [ ]:
def factor_probs(pk, pc) -> np.ndarray:
    pk = np.asarray(pk, float); pc = np.asarray(pc, float)
    if pk.shape != pc.shape or pk.ndim != 1: raise ValueError('Use matching 1-D probabilities.')
    if not np.isfinite(pk).all() or not np.isfinite(pc).all(): raise ValueError('Nonfinite probability.')
    if ((pk < 0) | (pk > 1) | (pc < 0) | (pc > 1)).any(): raise ValueError('Not probabilities.')
    return np.stack([pk * pc, pk * (1 - pc), (1 - pk) * (1 - pc), (1 - pk) * pc], axis=1)


### validate_features

In [ ]:
def validate_features(features: Mapping[str, np.ndarray], n: int) -> None:
    for k, x in features.items():
        if x.ndim != 2 or len(x) != n or not np.isfinite(x).all():
            raise ValueError(f'Invalid feature {k}: {x.shape}')


### load_e14_block_bank

In [ ]:
def load_e14_block_bank(path: str | Path, n: int) -> dict[str, np.ndarray]:
    with np.load(path, allow_pickle=False) as z:
        rows = z['rows'].astype(int)
        if not np.array_equal(np.sort(rows), np.arange(n)): raise ValueError('Invalid bank rows.')
        order = np.argsort(rows)
        return {'L13': z['BLOCK_LAST'][order].astype(np.float32)}


### render_prompt

In [ ]:
def render_prompt(tokenizer, question: str, context: str) -> dict:
    user = f'Question: {question}\nContext: {context}\nAnswer:'
    text = tokenizer.apply_chat_template([{'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user}], tokenize=False, add_generation_prompt=True)
    base = text.rfind(user)
    if base < 0: raise ValueError('Chat template changed user text.')
    qstart = base + len('Question: '); qend = qstart + len(question)
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    qpos = [i for i, (s, e) in enumerate(enc['offset_mapping']) if e > qstart and s < qend and e > s]
    if not qpos: raise ValueError('Question token span not found.')
    return {'ids': enc['input_ids'], 'qpos': qpos[-1],
            'prompt_sha256': hashlib.sha256(text.encode()).hexdigest()}


### load_e11_neutral

In [ ]:
def load_e11_neutral(folder: str | Path, f: pd.DataFrame, tokenizer=None) -> tuple[dict, dict]:
    """Load ONLY complete neutral Q-first shards; verify semantics and row identity."""
    folder = Path(folder)
    manifest_path = folder / 'manifest.json'
    if not manifest_path.exists(): raise FileNotFoundError(manifest_path)
    meta = json.loads(manifest_path.read_text())
    expected = {'rows': list(range(len(f))), 'protocol': 'e11-qfirst', 'directive': None,
                'collect': True, 'batch_size': 4, 'tag': 'neutral'}
    if meta != expected: raise ValueError('Not the original complete Q-first neutral manifest.')
    if tokenizer is None: raise ValueError('Tokenizer required to verify every cached prompt hash.')
    chunks = {k: [] for k in ['Q7', 'L7', 'Q13', 'L13']}; allrows = []; hashes = []
    for start in range(0, len(f), 4):
        npz = folder / f'batch_{start:06d}.npz'; js = npz.with_suffix('.json')
        if not npz.exists() or not js.exists(): raise FileNotFoundError(f'Incomplete neutral cache: {npz}')
        rec = json.loads(js.read_text())
        with np.load(npz, allow_pickle=False) as z:
            rows = z['rows'].astype(int)
            if not np.array_equal(rows, np.arange(start, min(start + 4, len(f)))):
                raise ValueError('Unexpected shard row ids.')
            if [int(r['row']) for r in rec] != rows.tolist(): raise ValueError('npz/json row mismatch.')
            for r in rec:
                i = int(r['row'])
                if r['action'] != 'noop' or float(r['max_applied_delta_ratio']) != 0:
                    raise ValueError('Steered features must not enter neutral state diagnostics.')
                want = render_prompt(tokenizer, str(f.iloc[i].question), str(f.iloc[i].ctx_text))['prompt_sha256']
                if r['prompt_sha256'] != want: raise ValueError(f'Cached prompt differs for row {i}.')
            for key in chunks: chunks[key].append(z[key].astype(np.float32))
            allrows.extend(rows.tolist())
        hashes.append({'name': npz.name, 'sha256': sha256_file(npz), 'json_sha256': sha256_file(js)})
    if allrows != list(range(len(f))): raise ValueError('Incomplete/duplicate features.')
    features = {k: np.concatenate(v) for k, v in chunks.items()}
    validate_features(features, len(f))
    return features, {'source': 'E11_original_neutral', 'manifest_sha256': sha256_file(manifest_path),
                      'all_prompt_hashes_verified': True, 'shards': hashes}


### feature_agreement

In [ ]:
def feature_agreement(x, ref) -> dict:
    x = np.asarray(x, np.float64); ref = np.asarray(ref, np.float64)
    if x.shape != ref.shape: raise ValueError('Feature bank dimension mismatch.')
    rel = np.linalg.norm(x - ref, axis=1) / np.maximum(np.linalg.norm(ref, axis=1), 1e-12)
    cos = np.sum(x * ref, axis=1) / np.maximum(np.linalg.norm(x, axis=1) * np.linalg.norm(ref, axis=1), 1e-12)
    return {'max_abs_difference': float(np.max(np.abs(x - ref))),
            'median_relative_L2': float(np.median(rel)), 'max_relative_L2': float(rel.max()),
            'median_cosine': float(np.median(cos)), 'min_cosine': float(cos.min())}


### question_invariance

In [ ]:
def question_invariance(f, features) -> dict:
    pairs = [g.row.to_numpy() for _, g in f.groupby('question', sort=True)]
    a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])
    return {k: feature_agreement(x[a], x[b]) for k, x in features.items() if k.startswith('Q')}


### fit_probe

In [ ]:
def fit_probe(x, y, fit_rows):
    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000,
        solver='lbfgs', random_state=42))
    with warnings.catch_warnings(), threadpool_limits(limits=1):
        warnings.simplefilter('error', ConvergenceWarning)
        m.fit(x[fit_rows], np.asarray(y)[fit_rows])
    return m


### _p1

In [ ]:
def _p1(model, x):
    col = np.flatnonzero(model.classes_ == 1)
    if len(col) != 1: raise ValueError('Binary probe is missing a class.')
    return model.predict_proba(x)[:, col[0]]


### run_cv

In [ ]:
def run_cv(f, features, out_dir, *, specs=SPECS, save_models=False) -> dict[str, pd.DataFrame]:
    """No test-dependent selection; all methods use exactly the E14 FIT partition."""
    import joblib
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    if not (out / 'analysis_plan.json').exists(): raise RuntimeError('Freeze the analysis plan first.')
    if (out / 'oof_predictions.csv').exists(): raise FileExistsError('Output exists. Use a new run directory.')
    locked = json.loads((out / 'analysis_plan.json').read_text(encoding='utf-8'))
    if locked['methods'] != json.loads(json.dumps([asdict(s) for s in specs])):
        raise ValueError('Requested methods differ from the frozen plan.')
    if locked['seed'] != 42 or locked['C_logreg'] != 0.1:
        raise ValueError('This implementation uses the fixed E14 seed and regularization.')
    validate_features(features, len(f))
    missing = {k for s in specs for k in s.keys} - set(features)
    if missing: raise KeyError(f'Missing neutral features: {sorted(missing)}. No synthetic substitution.')
    matrices = {s.name: np.concatenate([features[k] for k in s.keys], axis=1)
                for s in specs if s.mode == 'joint4'}
    pred = {s.name: np.full((len(f), 4), np.nan) for s in specs}
    fold_map = f.question.map(build_outer_map(f)).to_numpy()
    manifest = []; timings = []
    for fold in range(5):
        split = make_split(f, fold); fit = np.flatnonzero(split.role.eq('fit')); test = np.flatnonzero(split.role.eq('test'))
        split.to_csv(out / f'split_fold_{fold}.csv', index=False)
        # Fit identical binary reader only once per fold when reused across comparisons.
        binary = {}
        for spec in specs:
            start = time.perf_counter()
            if spec.mode == 'joint4':
                x = matrices[spec.name]; m = fit_probe(x, f.state, fit)
                if list(m.classes_) != [0, 1, 2, 3]: raise ValueError('Wrong state order.')
                p = m.predict_proba(x[test]); saved = {'model': m, 'keys': spec.keys, 'mode': spec.mode}
                dim = x.shape[1]
            elif spec.mode == 'factor':
                models = []
                for axis, key in zip(['k', 'c'], spec.keys):
                    if (axis, key) not in binary:
                        binary[(axis, key)] = fit_probe(features[key], f[axis], fit)
                    models.append(binary[(axis, key)])
                p = factor_probs(_p1(models[0], features[spec.keys[0]][test]),
                                 _p1(models[1], features[spec.keys[1]][test]))
                saved = {'models': models, 'keys': spec.keys, 'mode': spec.mode}
                dim = sum(features[k].shape[1] for k in set(spec.keys))
            else: raise ValueError(spec.mode)
            if not np.isnan(pred[spec.name][test]).all(): raise ValueError('Duplicate OOF prediction.')
            pred[spec.name][test] = p
            if save_models: joblib.dump(saved, out / f'fold_{fold}_{spec.name}.joblib')
            manifest.append({'fold': fold, 'method': spec.name, 'n_fit_rows': len(fit),
                'n_fit_questions': f.iloc[fit].question.nunique(), 'n_test_rows': len(test),
                'unique_input_dimensions': dim, 'uses_probe_at_runtime': True,
                'test_used_for_fit_or_selection': False})
            timings.append({'fold': fold, 'method': spec.name, 'seconds': time.perf_counter() - start})
        print(f'Fold {fold + 1}/5: all fixed readers predicted.', flush=True)
    frames = {}
    for name, p in pred.items():
        if not np.isfinite(p).all() or not np.allclose(p.sum(1), 1): raise ValueError('Incomplete probabilities.')
        d = f[['row', 'question', 'k', 'c', 'state']].copy().rename(columns={'state': 'true_state'})
        d['fold'] = fold_map; d['method'] = name; d['pred_state'] = p.argmax(1)
        for j in range(4): d[f'p{j}'] = p[:, j]
        frames[name] = d
    pd.concat(frames.values(), ignore_index=True).to_csv(out / 'oof_predictions.csv', index=False)
    pd.DataFrame(manifest).to_csv(out / 'fit_manifest.csv', index=False)
    pd.DataFrame(timings).to_csv(out / 'fit_timing.csv', index=False)
    write_json(out / 'environment.json', environment())
    return frames


### metrics4

In [ ]:
def metrics4(y, probs) -> dict:
    y = np.asarray(y); p = np.asarray(probs); pred = p.argmax(1)
    out = {'four_macro_F1': float(f1_score(y, pred, labels=[0,1,2,3], average='macro', zero_division=0)),
        'four_BAcc': float(balanced_accuracy_score(y, pred)), 'four_accuracy': float(accuracy_score(y, pred)),
        'NLL': float(log_loss(y, p, labels=[0,1,2,3])),
        'Brier4': float(np.mean(np.sum((p - np.eye(4)[y]) ** 2, axis=1)))}
    wrongs = {}
    for axis, pos in [('K', [0,1]), ('C', [0,3])]:
        truth = np.isin(y, pos).astype(int); score = p[:,pos].sum(1)
        out[axis + '_AUROC'] = float(roc_auc_score(truth, score))
        out[axis + '_BAcc'] = float(balanced_accuracy_score(truth, score >= .5))
        # Joint state errors use the joint argmax, not marginal threshold decisions.
        wrongs[axis] = np.isin(pred, pos) != truth.astype(bool)
    out['K_only_wrong'] = float(np.mean(wrongs['K'] & ~wrongs['C']))
    out['C_only_wrong'] = float(np.mean(wrongs['C'] & ~wrongs['K']))
    out['both_wrong'] = float(np.mean(wrongs['C'] & wrongs['K']))
    return out


### paired_bootstrap

In [ ]:
def paired_bootstrap(a: pd.DataFrame, b: pd.DataFrame, n_boot=2000, seed=42) -> dict:
    """Paired question-cluster bootstrap of fixed OOF predictions; no refitting."""
    a = a.sort_values('row').reset_index(drop=True); b = b.sort_values('row').reset_index(drop=True)
    for col in ['row', 'question', 'true_state', 'fold']:
        if not a[col].equals(b[col]): raise ValueError(f'Paired comparison mismatch: {col}')
    group, unique = pd.factorize(a.question, sort=True); g = len(unique)
    cm_a = np.zeros((g,16), np.int32); cm_b = cm_a.copy()
    np.add.at(cm_a, (group, a.true_state.to_numpy()*4 + a.pred_state.to_numpy()), 1)
    np.add.at(cm_b, (group, b.true_state.to_numpy()*4 + b.pred_state.to_numpy()), 1)
    def mf(v):
        m = v.reshape(-1,4,4); tp = np.diagonal(m, axis1=1, axis2=2)
        den = m.sum(1) + m.sum(2)
        return np.mean(np.divide(2*tp, den, out=np.zeros_like(tp,dtype=float), where=den>0), axis=1)
    rng = np.random.default_rng(seed); diffs = []
    for start in range(0, n_boot, 100):
        draw = rng.integers(0,g,size=(min(100,n_boot-start),g))
        diffs.extend((mf(cm_a[draw].sum(1)) - mf(cm_b[draw].sum(1))).tolist())
    point = float(mf(cm_a.sum(0))[0] - mf(cm_b.sum(0))[0])
    lo, hi = np.quantile(diffs, [.025,.975])
    return {'delta_macro_F1': point, 'ci95_lo': float(lo), 'ci95_hi': float(hi),
            'n_questions': g, 'bootstrap_replicates': n_boot,
            'scope': 'fixed_OOF_predictions_not_full_pipeline_or_new_dataset'}


### summarize

In [ ]:
def summarize(frames, out_dir, n_boot=2000) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = Path(out_dir); summaries=[]; folds=[]; perclass=[]
    for name, d in frames.items():
        p = d[[f'p{i}' for i in range(4)]].to_numpy()
        summaries.append({'method': name, **metrics4(d.true_state, p)})
        for fold, sub in d.groupby('fold'):
            folds.append({'method': name, 'fold': fold,
                          **metrics4(sub.true_state, sub[[f'p{i}' for i in range(4)]].to_numpy())})
        pr,re,fs,ns = precision_recall_fscore_support(d.true_state,d.pred_state,labels=[0,1,2,3],zero_division=0)
        for i in range(4): perclass.append({'method':name,'state':STATE_NAMES[i],
            'precision':pr[i],'recall':re[i],'F1':fs[i],'n':int(ns[i])})
        pd.DataFrame(confusion_matrix(d.true_state,d.pred_state,labels=[0,1,2,3]),
            index=STATE_NAMES,columns=STATE_NAMES).to_csv(out / f'confusion_{name}.csv')
    sm = pd.DataFrame(summaries); fd = pd.DataFrame(folds)
    fs = fd.groupby('method').four_macro_F1.agg(['mean','std']).rename(columns={'mean':'fold_F1_mean','std':'fold_F1_std'})
    sm = sm.merge(fs,left_on='method',right_index=True)
    rows=[]
    for a,b,kind in CONTRASTS:
        if a in frames and b in frames: rows.append({'comparison':kind,'A':a,'B':b,
            **paired_bootstrap(frames[a],frames[b],n_boot=n_boot)})
    contrast = pd.DataFrame(rows)
    sm.to_csv(out/'summary.csv',index=False); fd.to_csv(out/'fold_metrics.csv',index=False)
    pd.DataFrame(perclass).to_csv(out/'per_state_metrics.csv',index=False)
    contrast.to_csv(out/'paired_comparisons.csv',index=False)
    lines = ['# E18A 실행 결과', '', '실험 성격: 기존 p8s 개발셋의 후속 OOF 진단. 독립 확증이 아님.',
             'QA 생성/steering은 실행하지 않음. 아래 숫자는 답변 EM이 아니라 상태 분류 성능임.', '',
             '|방법|4-state Macro-F1|K AUROC|C AUROC|', '|---|---:|---:|---:|']
    for _,r in sm.iterrows(): lines.append(f"|{r['method']}|{r.four_macro_F1:.4f}|{r.K_AUROC:.4f}|{r.C_AUROC:.4f}|")
    lines += ['', '## Paired differences', 'CI는 질문 단위 fixed-OOF bootstrap이며 학습/설계 선택 불확실성 전체를 포함하지 않음.']
    for _,r in contrast.iterrows(): lines.append(f"\n{r.comparison}: {100*r.delta_macro_F1:+.2f} pp [{100*r.ci95_lo:+.2f}, {100*r.ci95_hi:+.2f}]")
    if 'M2_L7lastK_L13lastC' not in frames:
        lines += ['', '**L7 실험 미실행: 현재 결과로 early-K 가설의 성공/실패를 판단할 수 없음.**']
    lines += ['', '해석: 첫 질문 위치는 Q-first causal mask로 뒤 문맥을 볼 수 없다. 이것은 특정 early layer만의 성질이 아니다.',
        '좋은 state decoder가 좋은 QA router라는 보장은 없다. QA 검증은 별도 고정 actuator로 수행해야 한다.',
        '이 결과를 보고 최고 OOF 방법을 선택해 독립 test 성능으로 제시하지 않는다.']
    (out/'RESULTS_KO.md').write_text('\n'.join(lines),encoding='utf-8')
    return sm, contrast


### audit_reference

In [ ]:
def audit_reference(predictions: pd.DataFrame, reference_path) -> dict:
    a = predictions.sort_values('row').reset_index(drop=True)
    b = pd.read_csv(reference_path).sort_values('row').reset_index(drop=True)
    for col in ['row','fold','true_state']:
        if not np.array_equal(a[col], b[col]): raise ValueError(f'E14 reference mismatch: {col}')
    return {'n':len(a), 'same_predicted_state':int(np.sum(a.pred_state==b.pred_state)),
            'max_probability_difference':float(np.max(np.abs(a[[f'p{i}' for i in range(4)]].to_numpy()-
                                                            b[[f'p{i}' for i in range(4)]].to_numpy()))),
            'reference_macro_F1':float(f1_score(b.true_state,b.pred_state,average='macro')),
            'rerun_macro_F1':float(f1_score(a.true_state,a.pred_state,average='macro')),
            'environment':environment()}


In [ ]:
"""Frozen E18B: five question-layer representations with an archived C reader."""
from pathlib import Path
import gc, hashlib, json, shutil, time, warnings, zipfile
import numpy as np
import pandas as pd
import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score
from threadpoolctl import threadpool_limits

RUN_ID = 'E18B_qmultiK_fixedC_readable_v3'
LAYERS = [9, 13, 17, 21, 25]
B0 = 'B0_Q13_PCA80'
B1 = 'B1_Qmulti_PCA16x5'
ARMS = [B0, B1, 'X_rank1x5'] + [f'X_random16x5_s{s}' for s in [3,11,23]] + [f'X_Q{l}_PCA80' for l in LAYERS if l != 13]
OLD_BANK_SHA = '9ead3c0c2671bacf727253f95b27021e744ba30d1f2f29b893750b933fae652c'
PLAN = dict(run_id=RUN_ID, primary=[B1,B0], layers=LAYERS, methods=ARMS,
    model_id=MODEL_ID, model_revision=MODEL_REVISION, indexing='zero_based_block_output',
    neutral_prefill=True, batch_size=4, dtype='bfloat16', max_tokens=4096,
    pca=dict(center=True,whiten=False,svd_solver='full',seed=42,fit_role='fit'),
    scaling='after_projection; fit partition only', C_reader='archived E18A M1 OOF p0+p3',
    logreg=dict(C=.1,solver='lbfgs',max_iter=2000,class_weight=None,random_state=42),
    split='E14 fixed question-grouped 5 folds; 60/10/10/20; UF and PC unused',
    bootstrap=dict(unit='question',replicates=2000,seed=42,scope='fixed OOF predictions'),
    pass_rule=dict(primary_delta_at_least=.01,ci95_lower_above=0,
                   B1_at_least_E18A_M3=True,max_recall_drop_vs_M3=.03),
    exploratory='All rank/random/single-layer controls reported; no OOF winner selection',
    interpretation='supervised development diagnostic; no generation or steering; not independent confirmation')




### lock_json

In [ ]:
def lock_json(path, value):
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError(f'Frozen input/configuration changed: {path.name}')
    else:
        write_json(path, value)


### prepare

In [ ]:
def prepare(out, reference):
    out=Path(out); out.mkdir(parents=True,exist_ok=True); reference=Path(reference)
    lock_json(out/'analysis_plan.json',PLAN)
    lock_json(out/'implementation_hashes.json', {p.name:sha256_file(p) for p in [CODE_SNAPSHOT]})
    src=out/'inputs'; src.mkdir(exist_ok=True)
    import gdown
    paths={}
    for key in ['behavior','oof13']:
        spec=SOURCE_FILES[key]; p=src/spec['name']
        ensure_input(spec['id'], p, spec['sha256'])
        paths[key]=p
    f=load_labels(paths['behavior'],paths['oof13'])
    archived=pd.read_csv(reference/'oof_predictions.csv')
    anchors={}
    for name in ['M0_L13_joint4','M1_L13_factor','M3_L13qK_L13lastC']:
        d=archived.loc[archived.method.eq(name)].sort_values('row').reset_index(drop=True)
        for a,b in [('row','row'),('question','question'),('k','k'),('c','c'),('true_state','state')]:
            if not np.array_equal(d[a],f[b]): raise ValueError(f'Archived {name}: {a} mismatch')
        if not np.array_equal(d.fold,f.question.map(build_outer_map(f))): raise ValueError('Fold mismatch')
        anchors[name]=d
    for fold in range(5):
        sp=make_split(f,fold); old=pd.read_csv(reference/f'split_fold_{fold}.csv')
        pd.testing.assert_frame_equal(sp,old,check_dtype=False)
        sp.to_csv(out/f'split_fold_{fold}.csv',index=False)
    c=anchors['M1_L13_factor'][['p0','p3']].sum(axis=1).to_numpy()
    if not np.allclose(c, anchors['M3_L13qK_L13lastC'][['p0','p3']].sum(axis=1),rtol=0,atol=1e-14):
        raise ValueError('E18A C predictions not identical')
    lock_json(out/'input_lock.json',dict(source={k:sha256_file(v) for k,v in paths.items()},
        archived_oof_sha256=sha256_file(reference/'oof_predictions.csv'),
        frozen_C_sha256=hashlib.sha256(c.tobytes()).hexdigest(), old_bank_sha256=OLD_BANK_SHA))
    pd.DataFrame({'row':f.row,'fold':anchors['M1_L13_factor'].fold,'pc':c}).to_csv(out/'frozen_C.csv',index=False)
    print('Validated original labels, archived predictions, all five splits, and fixed C.',flush=True)
    return f, anchors, c


### forward_capture

In [ ]:
def forward_capture(model, tokenizer, pp):
    import torch
    dev=next(model.parameters()).device; t=max(len(x['ids']) for x in pp)
    ids=torch.full((len(pp),t),tokenizer.pad_token_id,dtype=torch.long,device=dev)
    mask=torch.zeros_like(ids); qi=[]; cache={}; handles=[]; counts={}
    for j,x in enumerate(pp):
        off=t-len(x['ids']); ids[j,off:]=torch.tensor(x['ids'],device=dev)
        mask[j,off:]=1; qi.append(off+x['qpos'])
    pos=(mask.cumsum(-1)-1).clamp_min(0); qi=torch.tensor(qi,device=dev)
    def hook(layer):
        def capture(mod,args,value):
            h=value[0] if isinstance(value,tuple) else value
            counts[layer]=counts.get(layer,0)+1
            cache[f'Q{layer}']=h[torch.arange(len(pp),device=dev),qi].detach().float().cpu().numpy().astype(np.float16)
            if layer==13: cache['L13']=h[:,-1].detach().float().cpu().numpy().astype(np.float16)
        return capture
    try:
        for l in LAYERS: handles.append(model.model.layers[l].register_forward_hook(hook(l)))
        with torch.inference_mode():
            value=model.model(input_ids=ids,attention_mask=mask,position_ids=pos,use_cache=False,return_dict=True)
        del value
        if counts!={l:1 for l in LAYERS}: raise ValueError('Unexpected number of block observations')
    finally:
        for h in handles: h.remove()
    return cache


### collect

In [ ]:
def collect(model, tokenizer, prompts, audit_rows, out, old_bank):
    import torch
    assert set(prompts.columns)=={'row','question','ctx_text'}
    assert np.array_equal(prompts.row,np.arange(len(prompts)))
    assert model.config._commit_hash==MODEL_REVISION
    assert next(model.parameters()).is_cuda and next(model.parameters()).dtype==torch.bfloat16
    assert 'A100' in torch.cuda.get_device_name(0)
    out=Path(out); capdir=out/'capture'; capdir.mkdir(exist_ok=True)
    pp=[render_prompt(tokenizer,str(r.question),str(r.ctx_text)) for r in prompts.itertuples()]
    assert max(len(x['ids']) for x in pp)<=4096, 'No truncation allowed'
    prefix=[hashlib.sha256(np.asarray(x['ids'][:x['qpos']+1],dtype=np.int64).tobytes()).hexdigest() for x in pp]
    audit=prompts[['row','question']].copy(); audit['prefix_sha256']=prefix
    audit['qpos']=[x['qpos'] for x in pp]; audit['length']=[len(x['ids']) for x in pp]
    assert audit.groupby('question').prefix_sha256.nunique().eq(1).all(), 'Question prefix mismatch'
    audit.to_csv(out/'prefix_audit.csv',index=False)
    manifest=dict(environment=environment(),gpu=torch.cuda.get_device_name(0),model=MODEL_ID,revision=MODEL_REVISION,
                  prompt_hashes=[x['prompt_sha256'] for x in pp],batch_size=4,dtype='bfloat16',layers=LAYERS,
                  position_ids='(left_pad_attention_mask.cumsum(-1)-1).clamp_min(0)')
    lock_json(capdir/'manifest.json',manifest)
    model.eval(); chunks=[]; shard_hashes={}
    for st in range(0,len(pp),4):
        p=capdir/f'batch_{st:06d}.npz'; rows=np.arange(st,min(st+4,len(pp)))
        if p.exists():
            with np.load(p,allow_pickle=False) as z: d={k:z[k].copy() for k in z.files}
        else:
            d={'rows':rows,**forward_capture(model,tokenizer,pp[st:st+4])}
            tmp=p.with_suffix('.tmp')
            with tmp.open('wb') as fh: np.savez_compressed(fh,**d)
            tmp.replace(p)
        assert np.array_equal(d['rows'],rows)
        for key in [f'Q{l}' for l in LAYERS]+['L13']:
            assert d[key].shape==(len(rows),4096) and np.isfinite(d[key]).all()
        shard_hashes[p.name]=sha256_file(p); chunks.append(d)
        if st%80==0: print(f'E18B capture {min(st+4,len(pp))}/{len(pp)}',flush=True)
    lock_json(capdir/'shard_hashes.json',shard_hashes)
    features={k:np.concatenate([d[k] for d in chunks]).astype(np.float32) for k in chunks[0] if k!='rows'}
    old_bank=Path(old_bank)
    if sha256_file(old_bank)!=OLD_BANK_SHA: raise ValueError('Archived feature bank changed')
    agreements={}
    with np.load(old_bank,allow_pickle=False) as old:
        assert np.array_equal(old['rows'],np.arange(len(pp)))
        for key in ['Q13','L13']: agreements[key]=feature_agreement(features[key],old[key].astype(np.float32))
    write_json(out/'E18A_feature_agreement.json',agreements)
    # Original capture was exact on A100. A mismatch requires an explicit environment investigation.
    if any(v['max_abs_difference']>0 for v in agreements.values()):
        raise ValueError('E18A feature mismatch; inspect agreement/environment before proceeding')
    singles={f'Q{l}':[] for l in LAYERS}
    for n,row in enumerate(audit_rows):
        p=capdir/f'audit_single_{row:06d}.npz'
        if not p.exists(): np.savez_compressed(p,**forward_capture(model,tokenizer,[pp[row]]))
        with np.load(p,allow_pickle=False) as z:
            for key in singles: singles[key].append(z[key][0])
        if n%16==0: print(f'Fit-only batch-size audit {n+1}/{len(audit_rows)}',flush=True)
    single={k:np.asarray(v,dtype=np.float32) for k,v in singles.items()}
    np.savez_compressed(capdir/'audit_single_features.npz',rows=audit_rows,**single)
    write_json(out/'batch_size_audit.json',{k:feature_agreement(features[k][audit_rows],single[k]) for k in single})
    np.savez_compressed(capdir/'features_all.npz',rows=np.arange(len(pp)),**{k:v.astype(np.float16) for k,v in features.items()})
    return features,single


### projected_views

In [ ]:
def projected_views(features, fit, fold, model_dir):
    """All PCA fits see fit rows only. Per-layer top-80 reused by nested rank controls."""
    pcs={}; z={}; reps={}; randoms={}
    for l in LAYERS:
        key=f'Q{l}'; x=features[key]
        pc=PCA(n_components=80,svd_solver='full',whiten=False,random_state=42).fit(x[fit])
        pcs[key]=pc; z[key]=pc.transform(x)
    reps[B0]=z['Q13']; reps[B1]=np.concatenate([z[f'Q{l}'][:,:16] for l in LAYERS],axis=1)
    reps['X_rank1x5']=np.concatenate([z[f'Q{l}'][:,:1] for l in LAYERS],axis=1)
    for l in LAYERS:
        if l!=13: reps[f'X_Q{l}_PCA80']=z[f'Q{l}']
    for seed in [3,11,23]:
        rng=np.random.default_rng(seed); mats={}; views=[]
        for l in LAYERS:
            w,_=np.linalg.qr(rng.standard_normal((features[f'Q{l}'].shape[1],16)))
            mats[f'Q{l}']=w.astype(np.float32); views.append(features[f'Q{l}']@mats[f'Q{l}'])
        randoms[seed]=mats; reps[f'X_random16x5_s{seed}']=np.concatenate(views,axis=1)
    joblib.dump(dict(pca=pcs,random_projection=randoms,fit_rows=fit),model_dir/f'projection_fold_{fold}.joblib',compress=3)
    return reps,pcs


### train_and_report

In [ ]:
def train_and_report(f,features,anchors,c,out,audit_rows,single):
    out=Path(out); md=out/'models';md.mkdir(exist_ok=True)
    preds={name:np.full((len(f),4),np.nan) for name in ARMS}; fitlog=[]; aud=[]
    foldmap=f.question.map(build_outer_map(f)).to_numpy()
    with threadpool_limits(limits=4), warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        for fold in range(5):
            sp=make_split(f,fold); fit=np.flatnonzero(sp.role.eq('fit')); test=np.flatnonzero(sp.role.eq('test'))
            reps,pcs=projected_views(features,fit,fold,md)
            for name in ARMS:
                t=time.time(); x=reps[name]
                probe=make_pipeline(StandardScaler(),LogisticRegression(**PLAN['logreg'])).fit(x[fit],f.k.iloc[fit])
                pk=probe.predict_proba(x[test])[:,list(probe.classes_).index(1)]
                preds[name][test]=factor_probs(pk,c[test])
                joblib.dump(probe,md/f'{name}_fold_{fold}.joblib',compress=3)
                fitlog.append(dict(method=name,fold=fold,n_fit_rows=len(fit),dimension=x.shape[1],seconds=time.time()-t))
                if fold==0 and name in [B0,B1]:
                    sx=pcs['Q13'].transform(single['Q13']) if name==B0 else np.concatenate([pcs[f'Q{l}'].transform(single[f'Q{l}'])[:,:16] for l in LAYERS],axis=1)
                    pa=probe.predict_proba(x[audit_rows])[:,1]; pb=probe.predict_proba(sx)[:,1]
                    for row,a,b in zip(audit_rows,pa,pb): aud.append(dict(method=name,row=int(row),batch4_pk=a,batch1_pk=b,abs_delta=abs(a-b),batch4_margin=abs(a-.5),decision_changed=bool((a>=.5)!=(b>=.5))))
            print(f'E18B fold {fold+1}/5 complete: {len(ARMS)} arms',flush=True)
    frames=dict(anchors)
    for name,p in preds.items():
        assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
        assert np.allclose(p[:,[0,3]].sum(axis=1),c,rtol=0,atol=1e-14)
        d=f[['row','question','k','c','state']].rename(columns={'state':'true_state'}).copy()
        d['fold']=foldmap; d['method']=name; d['pred_state']=p.argmax(axis=1)
        for i in range(4):d[f'p{i}']=p[:,i]
        frames[name]=d
    pd.concat(frames.values()).to_csv(out/'oof_predictions.csv',index=False)
    pd.DataFrame(fitlog).to_csv(out/'fit_manifest.csv',index=False)
    pd.DataFrame(aud).to_csv(out/'batch_size_prediction_audit.csv',index=False)
    sm=[]; folds=[]; perstate=[]
    for name,d in frames.items():
        p=d[[f'p{i}' for i in range(4)]].to_numpy(); pk=p[:,:2].sum(axis=1)
        pair=pd.DataFrame({'question':d.question,'decision':pk>=.5}).groupby('question').decision.nunique()
        row=dict(method=name,role='primary' if name in [B0,B1] else ('anchor' if name in anchors else 'exploratory'),
                 **metrics4(d.true_state,p),K_Brier=float(np.mean((pk-d.k.to_numpy())**2)),K_pair_flips=int((pair>1).sum()))
        for cv in [0,1]:
            sel=d.c.to_numpy()==cv;row[f'K_BAcc_C{cv}']=float(balanced_accuracy_score(d.k.to_numpy()[sel],pk[sel]>=.5))
        sm.append(row)
        for fold,sub in d.groupby('fold'):folds.append(dict(method=name,fold=int(fold),**metrics4(sub.true_state,sub[[f'p{i}' for i in range(4)]].to_numpy())))
        pr,re,fs,ns=precision_recall_fscore_support(d.true_state,d.pred_state,labels=[0,1,2,3],zero_division=0)
        perstate.extend(dict(method=name,state=STATE_NAMES[i],precision=pr[i],recall=re[i],F1=fs[i],n=int(ns[i])) for i in range(4))
    summary=pd.DataFrame(sm);fd=pd.DataFrame(folds);ps=pd.DataFrame(perstate)
    summary=summary.merge(fd.groupby('method').four_macro_F1.agg(fold_F1_mean='mean',fold_F1_std='std'),left_on='method',right_index=True)
    contrasts=[]
    for a,b,kind in [(B1,B0,'PRIMARY'),(B1,'M3_L13qK_L13lastC','vs_E18A_M3'),(B0,'M3_L13qK_L13lastC','compression_effect')]:
        contrasts.append(dict(A=a,B=b,kind=kind,**paired_bootstrap(frames[a],frames[b],2000,42)))
    cr=pd.DataFrame(contrasts);primary=contrasts[0]
    point=summary.set_index('method').four_macro_F1
    recalls=ps.pivot(index='state',columns='method',values='recall')
    maxdrop=float((recalls['M3_L13qK_L13lastC']-recalls[B1]).max())
    checks=dict(delta_at_least_1pp=bool(primary['delta_macro_F1']>=.01),CI_lower_positive=bool(primary['ci95_lo']>0),
                at_least_E18A_M3=bool(point[B1]>=point['M3_L13qK_L13lastC']),recall_drop_at_most_3pp=bool(maxdrop<=.03))
    gate=dict(passed=all(checks.values()),checks=checks,max_state_recall_drop_vs_M3=maxdrop,
              interpretation='Meets engineering adoption rule' if all(checks.values()) else 'Does not meet prespecified adoption rule; no winner selection or parameter sweep')
    for fn,df in [('summary.csv',summary),('fold_metrics.csv',fd),('per_state_metrics.csv',ps),('paired_comparisons.csv',cr)]:df.to_csv(out/fn,index=False)
    write_json(out/'decision.json',gate);write_json(out/'environment.json',environment())
    lines=['# E18B 실행 결과','',PLAN['interpretation'],'','|방법|OOF 4-state Macro-F1|K AUROC|K pair flips|','|---|---:|---:|---:|']
    for r in summary.itertuples():lines.append(f'|{r.method}|{r.four_macro_F1:.6f}|{r.K_AUROC:.6f}|{r.K_pair_flips}/612|')
    lines+=['',f"Primary B1−B0: {primary['delta_macro_F1']*100:+.3f} pp; 95% question-bootstrap CI [{primary['ci95_lo']*100:+.3f}, {primary['ci95_hi']*100:+.3f}] pp.",'',f'Adoption rule passed: {gate["passed"]}. {checks}', '', 'CI는 고정 OOF 예측의 질문 단위 bootstrap이다. 학습·설계 선택 전체의 불확실성을 포함하지 않는다. 탐색 대조군을 보고 최고 OOF 방법을 선택하지 않는다.']
    (out/'RESULTS_KO.md').write_text('\n'.join(lines),encoding='utf-8')
    archive=out.parent/f'{RUN_ID}_results.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.iterdir()):
            if p.is_file():z.write(p,p.name)
        for p in [CODE_SNAPSHOT]:z.write(p,'code/'+p.name)
    print(summary[['method','four_macro_F1','K_AUROC','K_pair_flips']].to_string(index=False),flush=True)
    print(json.dumps(gate,ensure_ascii=False,indent=2));print('RESULTS_ZIP',archive,sha256_file(archive),flush=True)
    return summary,cr,gate


## 5. 코드 기록 및 기준 결과 자동 다운로드
기존 E18A 결과 ZIP을 자동 다운로드하고 SHA-256을 검증합니다.

In [ ]:
CODE = Path('/content/e18b_readable_code')
CODE.mkdir(exist_ok=True)
CODE_SNAPSHOT = CODE / 'e18b_readable_definitions.py'
_definition_end = get_ipython().execution_count
# 함수 셀의 실제 실행 소스를 기록합니다. 인증값은 저장하지 않습니다.
_source = '\n\n'.join([*In[_definition_start:_definition_end], In[_input_setup_cell]])
CODE_SNAPSHOT.write_text(_source, encoding='utf-8')
archive_path = ensure_input(
    '16gsGWwRrFwTs-vg_v7qRf28eEW6PhERy',
    INPUT_CACHE / 'E18A_earlyK_lateC_v1_results.zip',
    '57a9ffd1592100b4507ac2d953807b1ac43c0b2a75febf962702bcd17fb9af98',
    local_candidates=['/content/drive/MyDrive/p8s/E18A_earlyK_lateC_v1_results.zip'],
)
REF = CODE / 'reference'
REF.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as z:
    assert all('/' not in name and '..' not in name for name in z.namelist())
    z.extractall(REF)
print('기준 결과 검증 완료')


## 6. 데이터 준비 실행 — 라벨·5-fold·고정 C 확률 검증

In [ ]:
OUT = Path('/content/drive/MyDrive/p8s') / RUN_ID
f, anchors, fixed_c = prepare(OUT, REF)
shutil.copyfile(CODE_SNAPSHOT, OUT / CODE_SNAPSHOT.name)
sp0 = make_split(f, 0)
fit_questions = sorted(sp0.loc[sp0.role.eq('fit'), 'question'].unique())
audit_questions = np.random.default_rng(42).choice(fit_questions, 32, replace=False)
audit_rows = f.loc[f.question.isin(audit_questions), 'row'].to_numpy()
assert len(audit_rows) == 64 and sp0.iloc[audit_rows].role.eq('fit').all()
write_json(OUT/'audit_selection.json', {'fold':0, 'role':'fit', 'seed':42, 'rows':audit_rows.tolist()})
print('데이터 준비 완료:', len(f), '행 /', f.question.nunique(), '질문')
print('결과 저장 위치:', OUT)


## 7. Hugging Face 인증
토큰 입력을 완료한 다음 다음 셀을 실행하세요. 토큰을 코드에 쓰지 마세요.

In [ ]:
from huggingface_hub import get_token, notebook_login
if get_token() is None:
    notebook_login()
else:
    print('Existing Hugging Face credential available (not displayed).')


## 8. 모델 로딩

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import get_token
assert get_token() is not None, 'Complete the secure Hugging Face login widget, then rerun this cell and following cells.'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,
    torch_dtype=torch.bfloat16,attn_implementation='sdpa',device_map={'':'cuda:0'})
model.eval()


## 9. 특징 추출 실행 — Q9·Q13·Q17·Q21·Q25 및 기존 표현 일치 검사

In [ ]:
features,single_features=collect(model,tokenizer,f[['row','question','ctx_text']],audit_rows,OUT,OLD_BANK)
del model
gc.collect(); torch.cuda.empty_cache()
print('Capture and E18A feature parity validated.')


## 10. 5-fold 학습 및 OOF 평가 실행
B0/B1 및 8개 대조군을 모두 평가합니다. PCA와 정규화는 각 fold의 fit 구간에서만 학습합니다. 질문 단위 bootstrap과 사전 판정 기준도 함께 계산합니다.

In [ ]:
summary,comparisons,decision=train_and_report(f,features,anchors,fixed_c,OUT,audit_rows,single_features)
display(summary)
display(comparisons)
print('COMPLETE', OUT)


## 11. 결과 요약 및 ZIP 다운로드

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUT/'RESULTS_KO.md').read_text()))
print('Result archive:', OUT.parent/f'{RUN_ID}_results.zip')

from google.colab import files
files.download(str(OUT.parent / f'{RUN_ID}_results.zip'))
